In [1]:
import pandas as pd

# Step 1: Data Load လုပ်ခြင်း (Data Loading for Language Identification)
df = pd.read_csv('dataset.csv')
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print("\nUnique Languages:", df['language'].nunique())
print(df['language'].value_counts().head(10))

Dataset Shape: (22000, 2)
Columns: ['Text', 'language']

First 5 rows:
                                                Text  language
0  klement gottwaldi surnukeha palsameeriti ning ...  Estonian
1  sebes joseph pereira thomas  på eng the jesuit...   Swedish
2  ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...      Thai
3  விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...     Tamil
4  de spons behoort tot het geslacht haliclona en...     Dutch

Unique Languages: 22
language
Estonian      1000
Swedish       1000
Thai          1000
Tamil         1000
Dutch         1000
Japanese      1000
Turkish       1000
Latin         1000
Urdu          1000
Indonesian    1000
Name: count, dtype: int64


In [2]:
# Step 2: Feature & Target Selection
X = df['Text']      # Text column
y = df['language']  # Language label (22 classes)

In [3]:
from sklearn.model_selection import train_test_split

# Step 3: Train/Test ခွဲခြင်း
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

Train size: 17600 Test size: 4400


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Step 4: စာသားများကို ကိန်းဂဏန်းအဖြစ် ပြောင်းလဲခြင်း (Text Vectorization with TF-IDF)
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Train shape:", X_train_tfidf.shape)

TF-IDF Train shape: (17600, 5000)


In [5]:
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
import pandas as pd

# Step 5: မော်ဒယ်များစွာ သုံး၍ တိကျမှု စစ်ဆေးခြင်း (Train All Models)
models_to_try = {
    "ComplementNB": ComplementNB(alpha=0.1),
    "MultinomialNB": MultinomialNB(alpha=0.1),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC()
}

fitted_models = {}
accuracies = {}

for name, model in models_to_try.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    accuracies[name] = acc
    fitted_models[name] = model
    print(f"{name} Accuracy: {acc:.4f}")

# Show results as table (as requested)
results_df = pd.DataFrame(list(accuracies.items()), columns=['Model', 'Accuracy'])
results_df = results_df.sort_values(by='Accuracy', ascending=False)
print("\n=== ACCURACY RESULTS (sorted) ===")
print(results_df)

ComplementNB Accuracy: 0.9155
MultinomialNB Accuracy: 0.9157
LogisticRegression Accuracy: 0.9436
LinearSVC Accuracy: 0.9461

=== ACCURACY RESULTS (sorted) ===
                Model  Accuracy
3           LinearSVC  0.946136
2  LogisticRegression  0.943636
1       MultinomialNB  0.915682
0        ComplementNB  0.915455


In [6]:
# Step 6: Choose the best one
best_model_name = max(accuracies, key=accuracies.get)
best_acc = accuracies[best_model_name]
best_model = fitted_models[best_model_name]

print(f"\n🎯 BEST MODEL: {best_model_name}")
print(f"Accuracy: {best_acc:.4f}")
print("We will use this model for prediction below.")


🎯 BEST MODEL: LinearSVC
Accuracy: 0.9461
We will use this model for prediction below.


In [7]:
# Step 7: စာသားအသစ်များဖြင့် လက်တွေ့စမ်းသပ်ခြင်း (Manual Testing with Best Model)
sample_tweets = [
    "I love this airline! The service was excellent and the flight was smooth.",  # English
    "Bonjour, comment allez-vous aujourd'hui ?",                                 # French
    "Hola, ¿cómo estás? El vuelo fue muy bueno.",                               # Spanish
    "مرحبا، كيف حالك اليوم؟",                                                  # Arabic
    "नमस्ते, आज का मौसम बहुत अच्छा है।",                                        # Hindi
    "こんにちは、今日はとてもいい天気です。"                                     # Japanese
]

sample_vec = tfidf.transform(sample_tweets)
predictions = best_model.predict(sample_vec)

for text, pred in zip(sample_tweets, predictions):
    print(f"Text: {text[:80]}...")
    print(f"Predicted Language: {pred}\n")

Text: I love this airline! The service was excellent and the flight was smooth....
Predicted Language: English

Text: Bonjour, comment allez-vous aujourd'hui ?...
Predicted Language: Chinese

Text: Hola, ¿cómo estás? El vuelo fue muy bueno....
Predicted Language: Spanish

Text: مرحبا، كيف حالك اليوم؟...
Predicted Language: Chinese

Text: नमस्ते, आज का मौसम बहुत अच्छा है।...
Predicted Language: Hindi

Text: こんにちは、今日はとてもいい天気です。...
Predicted Language: Chinese

